# Chapter 10, Application 2: Trotter-free simulation by qubitization

Companion notebook to *Quantum Algorithms from First Principles*, Chapter 10,
"Building a qubitized simulator" and "Testing a qubitized simulator".

Steps: (1) block-encode $H=\sum_j c_jP_j$ by LCU with the signs absorbed into the
select operator, $\alpha=\|c\|_1$; (2) the evolution time becomes $\alpha t$; (3) Jacobi–Anger
coefficients $J_k(\alpha t)$ truncated at $K$, split by parity into the $\cos(\alpha t\,x)$
and $-i\sin(\alpha t\,x)$ halves, phase factors found offline for each; (4) two QSVT
circuits added by a one-ancilla LCU with Hadamards on either side, delivering
$\tfrac12e^{-iHt}$.

Tests, in the book's order: block encoding equals $H/\alpha$; walk eigenvalues
$e^{\pm i\arccos(\lambda_j/\alpha)}$; $W^d$ projects to $T_d(H/\alpha)$; only then the full
circuit against `expm(-1j*H*t)`.

The phase-finding routine is a small optimizer written in the book's own convention
(reflection-type block encoding, $\Pi_\phi=e^{i\phi(2\Pi-I)}$). For degrees beyond a few
dozen use an established package, as the book advises; verify its output the same way.

Convention: book wire $j$ is Qiskit qubit `bq(j)`; Pauli strings in book order.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
from scipy.optimize import minimize
from scipy.special import jv
from numpy.polynomial import chebyshev as Ch
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, SparsePauliOp

bq = lambda j, tot: tot - 1 - j

## The Hamiltonian and its LCU block encoding

$H = 0.5\,XI - 0.3\,ZZ + 0.2\,IY + 0.4\,XX$ on two qubits: four Pauli terms, one with a
negative sign, $\alpha=1.4$. Two index ancillas; $O$ is the binary-tree preparation of
Chapter 7, $V$ applies $\mathrm{sign}(c_j)P_j$ controlled on $|j\rangle$. Because $O$ is real and
$V$ is Hermitian and self-inverse, $U=O^\dagger VO$ is Hermitian with $U^2=I$, which is the
self-inverse encoding the walk operator wants.

In [2]:
terms = [("XI", 0.5), ("ZZ", -0.3), ("IY", 0.2), ("XX", 0.4)]
n_sys = 2
coeffs = np.array([abs(c) for _, c in terms])
alpha = coeffs.sum()
H = sum(c * SparsePauliOp(P).to_matrix() for P, c in terms)
a = int(np.ceil(np.log2(len(terms))))            # index ancillas
print(f"alpha = ||c||_1 = {alpha}, eigenvalues of H/alpha:", np.round(np.linalg.eigvalsh(H) / alpha, 4))


def prepare_tree(weights, qc, qubits):
    k = len(qubits)
    w = np.zeros(2**k); w[:len(weights)] = weights
    for depth in range(k):
        block = 2**(k - depth)
        for node in range(2**depth):
            lo = node * block; tot_w = w[lo:lo + block].sum()
            if tot_w == 0: continue
            angle = 2 * np.arccos(np.sqrt(w[lo:lo + block // 2].sum() / tot_w))
            if depth == 0:
                qc.ry(angle, qubits[0])
            else:
                prefix = [(node >> (depth - 1 - i)) & 1 for i in range(depth)]
                zeros = [q for q, bit in zip(qubits[:depth], prefix) if bit == 0]
                if zeros: qc.x(zeros)
                qc.mcry(angle, qubits[:depth], qubits[depth])
                if zeros: qc.x(zeros)


def block_encoding():
    """U = O^dagger V O on a + n_sys qubits: ancillas = book wires 0..a-1, system = a..a+n_sys-1."""
    tot = a + n_sys
    idx = [bq(j, tot) for j in range(a)]
    sysq = [bq(a + j, tot) for j in range(n_sys)]
    O = QuantumCircuit(tot); prepare_tree(coeffs, O, idx)
    qc = QuantumCircuit(tot, name="U_BE")
    qc.compose(O, inplace=True)
    for i, (P, c) in enumerate(terms):
        bits = [(i >> (a - 1 - d)) & 1 for d in range(a)]
        zeros = [q for q, bit in zip(idx, bits) if bit == 0]
        if zeros: qc.x(zeros)
        g = SparsePauliOp(P, coeffs=[np.sign(c)]).to_operator().to_instruction().control(a)
        qc.append(g, idx + sysq[::-1])
        if zeros: qc.x(zeros)
    qc.compose(O.inverse(), inplace=True)
    return qc


U_be = block_encoding()
U = Operator(U_be).data
N = 2**n_sys
blk = U[:N, :N]                                      # ancillas = 0 occupy the top-left block (ancillas are the top wires)
print("Test 1: block = H/alpha:", np.allclose(blk, H / alpha), " | U Hermitian:", np.allclose(U, U.conj().T), " U^2 = I:", np.allclose(U @ U, np.eye(len(U))))
assert np.allclose(blk, H / alpha)

alpha = ||c||_1 = 1.4, eigenvalues of H/alpha: [-0.7311 -0.1287  0.1287  0.7311]
Test 1: block = H/alpha: True  | U Hermitian: True  U^2 = I: True


## Test 2: the walk operator $W=Z_\Pi U$ has eigenvalues $e^{\pm i\arccos(\lambda_j/\alpha)}$

In [3]:
Pi = np.kron(np.diag([1] + [0] * (2**a - 1)), np.eye(N))        # projector onto ancillas = 0
Z_Pi = 2 * Pi - np.eye(len(U))
W = Z_Pi @ U
ev = np.linalg.eigvals(W)
lam = np.linalg.eigvalsh(H) / alpha
expected = np.concatenate([np.exp(1j * np.arccos(lam)), np.exp(-1j * np.arccos(lam))])
# W also acts on the part of the ancilla space outside the invariant subspaces; keep the eigenvalues that match
matched = [np.min(np.abs(ev - e)) for e in expected]
print("walk eigenvalues matching e^{±i arccos(lambda/alpha)} to", f"{max(matched):.1e}")
assert max(matched) < 1e-10

walk eigenvalues matching e^{±i arccos(lambda/alpha)} to 1.3e-14


## Test 3: $W^d$ projects to $T_d(H/\alpha)$

That is the QSVT circuit with every phase equal to $\pi/2$, and it isolates qubitization
bugs from phase-factor bugs.

In [4]:
w_, V_ = np.linalg.eigh(H / alpha)
for d in range(1, 7):
    Wd = np.linalg.matrix_power(W, d)[:N, :N]
    # Chebyshev polynomial of a matrix via its eigendecomposition
    Td_A = V_ @ np.diag(Ch.chebval(w_, [0] * d + [1])) @ V_.conj().T
    err = np.linalg.norm(Wd - Td_A, 2)
    print(f"d={d}: ||<0|W^d|0> - T_d(H/alpha)|| = {err:.1e}")
    assert err < 1e-10

d=1: ||<0|W^d|0> - T_d(H/alpha)|| = 1.1e-14
d=2: ||<0|W^d|0> - T_d(H/alpha)|| = 2.3e-14
d=3: ||<0|W^d|0> - T_d(H/alpha)|| = 2.8e-14
d=4: ||<0|W^d|0> - T_d(H/alpha)|| = 5.2e-14
d=5: ||<0|W^d|0> - T_d(H/alpha)|| = 5.3e-14
d=6: ||<0|W^d|0> - T_d(H/alpha)|| = 5.2e-14


## The QSVT circuit and the phase finder

$\Pi_\phi = e^{i\phi(2\Pi-I)}$ on the $a$ ancillas: a phase $e^{2i\phi}$ on $|0\cdots0\rangle$ times the
global $e^{-i\phi}$. The circuit is $\Pi_{\phi_0}\prod_k[U\,\Pi_{\phi_k}]$ as an operator product,
which in circuit order means $\Pi_{\phi_d}$ first. Phases are found by least squares on
the scalar version of the same product with $U(x)=\begin{pmatrix}x&s\\s&-x\end{pmatrix}$,
constrained symmetric, so that $\mathrm{Re}\,P(x)$ matches the target.

In [5]:
def Pi_phi(qc, idx, phi):
    """e^{i phi (2 Pi - I)} on the ancilla list idx."""
    qc.x(idx)
    if len(idx) == 1:
        qc.p(2 * phi, idx[0])
    else:
        qc.mcp(2 * phi, idx[:-1], idx[-1])
    qc.x(idx)
    qc.global_phase += -phi


def qsvt_circuit(U_gate, phases, tot, idx):
    qc = QuantumCircuit(tot)
    for phi in reversed(phases[1:]):          # operator product read right to left
        Pi_phi(qc, idx, phi)
        qc.append(U_gate, range(tot))
    Pi_phi(qc, idx, phases[0])
    return qc


def qsvt_scalar_vec(phases, xs):
    """Vectorized <0|Pi_{phi0} prod[U(x) Pi_{phik}]|0> for the 1x1 'A = x' case."""
    xs = np.asarray(xs, dtype=float); s = np.sqrt(1 - xs**2)
    Ux = np.stack([np.stack([xs, s], -1), np.stack([s, -xs], -1)], -2)   # (m,2,2)
    D = lambda p: np.array([[np.exp(1j * p), 0], [0, np.exp(-1j * p)]])
    M = np.broadcast_to(D(phases[0]), Ux.shape).copy()
    for p in phases[1:]:
        M = M @ Ux @ D(p)
    return M[:, 0, 0]


def find_phases(target, d, n_restarts=6, seed=0):
    xs = np.cos(np.pi * (np.arange(2 * d + 2) + 0.5) / (2 * d + 2))
    ts = target(xs)
    half = (d + 2) // 2
    expand = (lambda h: np.concatenate([h, h[::-1]])) if d % 2 == 1 else (lambda h: np.concatenate([h, h[-2::-1]]))
    loss = lambda h: np.sum((np.real(qsvt_scalar_vec(expand(h), xs)) - ts) ** 2)
    rng = np.random.default_rng(seed); best = None
    for h0 in [np.full(half, np.pi / 2)] + [rng.uniform(0, np.pi, half) for _ in range(n_restarts)]:
        r = minimize(loss, h0, method="L-BFGS-B", options=dict(ftol=1e-16, gtol=1e-12, maxiter=5000))
        if best is None or r.fun < best.fun: best = r
        if best.fun < 1e-20: break
    return expand(best.x), best.fun


# sanity: all pi/2 reproduces T_d up to the phase i^{d+1}, in both the scalar model and the circuit
tot = a + n_sys
idx = [bq(j, tot) for j in range(a)]
U_gate = U_be.to_gate()
for d in [2, 3]:
    ph = np.full(d + 1, np.pi / 2)
    xs = np.linspace(-0.9, 0.9, 5)
    assert np.allclose(qsvt_scalar_vec(ph, xs), 1j**(d + 1) * Ch.chebval(xs, [0] * d + [1]))
    blk_d = Operator(qsvt_circuit(U_gate, ph, tot, idx)).data[:N, :N]
    w_, V_ = np.linalg.eigh(H / alpha)
    assert np.allclose(blk_d, 1j**(d + 1) * V_ @ np.diag(Ch.chebval(w_, [0] * d + [1])) @ V_.conj().T)
print("all-pi/2 phases give i^{d+1} T_d in the scalar model and in the circuit")

all-pi/2 phases give i^{d+1} T_d in the scalar model and in the circuit


## Step 3: Jacobi–Anger coefficients, truncated, split by parity

In [6]:
t = 1.5
tau = alpha * t                                   # the evolution time the circuit sees
K = 12                                            # truncation order; the error below says whether it is enough
cos_coef = np.zeros(K + 1); sin_coef = np.zeros(K + 1)
cos_coef[0] = jv(0, tau)
for k in range(1, K + 1):
    if k % 2 == 0: cos_coef[k] = 2 * (-1) ** (k // 2) * jv(k, tau)
    else:          sin_coef[k] = 2 * (-1) ** ((k - 1) // 2) * jv(k, tau)
xs = np.linspace(-1, 1, 401)
err_cos = np.max(np.abs(Ch.chebval(xs, cos_coef) - np.cos(tau * xs)))
err_sin = np.max(np.abs(Ch.chebval(xs, sin_coef) - np.sin(tau * xs)))
print(f"alpha t = {tau:.2f}, K = {K}: truncation error cos {err_cos:.1e}, sin {err_sin:.1e}")
# |target| must stay below 1 for the phase finder; scale both halves by a common factor
scale = 0.9
f_cos = lambda x: scale * Ch.chebval(x, cos_coef)
f_sin = lambda x: scale * Ch.chebval(x, sin_coef)
d_even = K if K % 2 == 0 else K - 1
d_odd = K - 1 if K % 2 == 0 else K
ph_cos, res_c = find_phases(f_cos, d_even)
ph_sin, res_s = find_phases(f_sin, d_odd)
print(f"phase fit residuals: cos {res_c:.1e} (degree {d_even}), sin {res_s:.1e} (degree {d_odd})")
print("max |Re P - target|: cos", f"{np.max(np.abs(np.real(qsvt_scalar_vec(ph_cos, xs)) - f_cos(xs))):.1e}",
      " sin", f"{np.max(np.abs(np.real(qsvt_scalar_vec(ph_sin, xs)) - f_sin(xs))):.1e}")

alpha t = 2.10, K = 12: truncation error cos 4.2e-11, sin 5.6e-10


phase fit residuals: cos 1.2e-11 (degree 12), sin 3.2e-12 (degree 11)
max |Re P - target|: cos 1.2e-06  sin 5.2e-07


## Step 4: the full circuit

Each half needs the real-part trick (one extra ancilla, controlled $U_{\boldsymbol\phi}$ and
$U_{-\boldsymbol\phi}$ between Hadamards) because the phase finder matches only
$\mathrm{Re}\,P$. The two halves are then combined by a one-ancilla LCU with coefficients
$1$ and $-i$: $\cos(\tau x) - i\sin(\tau x) = e^{-i\tau x}$. Register: 1 (combine) + 1
(real part) + $a$ (block encoding) + $n$ (system).

In [7]:
def real_part_block(phases):
    """Circuit on 1 + a + n qubits whose (all ancillas = 0) block is Re P(H/alpha)."""
    t2 = 1 + tot
    c = bq(0, t2); rest = list(range(t2 - 1))         # Qiskit qubits 0..tot-1 hold the block-encoding circuit
    qc = QuantumCircuit(t2)
    Up = qsvt_circuit(U_gate, phases, tot, idx).to_gate(label="U_phi")
    Um = qsvt_circuit(U_gate, -np.array(phases), tot, idx).to_gate(label="U_-phi")
    qc.h(c)
    qc.x(c); qc.append(Up.control(1), [c] + rest); qc.x(c)
    qc.append(Um.control(1), [c] + rest)
    qc.h(c)
    return qc


def simulator_circuit(ph_cos, ph_sin):
    t3 = 2 + tot
    sel = bq(0, t3); rest = list(range(t3 - 1))
    qc = QuantumCircuit(t3)
    Ce = real_part_block(ph_cos).to_gate(label="cos")
    So = real_part_block(ph_sin).to_gate(label="sin")
    qc.h(sel)
    qc.x(sel); qc.append(Ce.control(1), [sel] + rest); qc.x(sel)
    qc.append(So.control(1), [sel] + rest)
    qc.p(-np.pi / 2, sel)                     # the -i on the sin branch, before the final Hadamard
    qc.h(sel)
    return qc


qc_sim = simulator_circuit(ph_cos, ph_sin)
U_sim = Operator(qc_sim).data
blk_sim = U_sim[:N, :N]
target = scale * 0.5 * expm(-1j * H * t)      # LCU delivers 1/2, the phase finder scale another 0.9
err = np.linalg.norm(blk_sim - target, 2)
print(f"Test 4: ||block - (scale/2) e^{{-iHt}}|| = {err:.1e}   (qubits: {qc_sim.num_qubits}, uses of U per half: {d_even}, {d_odd})")
assert err < 1e-5
# the output state on post-selection, compared with exact evolution of a test state
psi = np.array([1, 0, 0, 0], dtype=complex); psi /= np.linalg.norm(psi)
out = blk_sim @ psi; out /= np.linalg.norm(out)
ex = expm(-1j * H * t) @ psi
print("fidelity with exact e^{-iHt}|psi>:", f"{abs(np.vdot(ex, out))**2:.8f}", "  success probability", f"{np.linalg.norm(blk_sim @ psi)**2:.4f} (≈ (scale/2)^2 = {(scale/2)**2:.4f})")

Test 4: ||block - (scale/2) e^{-iHt}|| = 2.7e-07   (qubits: 6, uses of U per half: 12, 11)
fidelity with exact e^{-iHt}|psi>: 1.00000000   success probability 0.2025 (≈ (scale/2)^2 = 0.2025)


## Cost against $\alpha t$: the Jacobi–Anger truncation grows linearly, as eq. (K) says

In [8]:
for tau_ in [0.5, 1, 2, 4, 8]:
    k = 0
    while 2 * abs(jv(k + 1, tau_)) > 1e-8: k += 1
    print(f"alpha t = {tau_:4.1f}: K for 1e-8 truncation = {k}")

alpha t =  0.5: K for 1e-8 truncation = 7
alpha t =  1.0: K for 1e-8 truncation = 9
alpha t =  2.0: K for 1e-8 truncation = 11
alpha t =  4.0: K for 1e-8 truncation = 15
alpha t =  8.0: K for 1e-8 truncation = 22
